[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/23_probabilistic_machine_learning/02_gmm_and_em.ipynb)

# Bài 02 — Gaussian Mixture Model và thuật toán EM

**🎯 Mục tiêu**
- GMM như một **mô hình sinh** với biến ẩn; vì sao không có nghiệm đóng.
- **EM from scratch**: E-step (responsibility), M-step; log-likelihood tăng đơn điệu; trực giác về cận dưới (ELBO).
- Khởi tạo và cực trị địa phương; chọn số thành phần bằng **BIC**; các kiểu hiệp phương sai.
- Failure case: suy biến (singularity); GMM vs K-Means; GMM để ước lượng mật độ, sinh dữ liệu và phát hiện bất thường.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/23_probabilistic_machine_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy.special import logsumexp
from scipy.stats import multivariate_normal
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture

from ml_course.algorithms import GaussianMixtureScratch
from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Khi K-Means không đủ

Ba nhóm khách hàng có **hình dạng và kích thước khác nhau** (một nhóm dẹt và nghiêng, một nhóm nhỏ, một nhóm lớn). K-Means giả định cụm tròn, cùng cỡ (chương 20):

In [ ]:
true_means = np.array([[0, 0], [5, 3], [6, -2]])
true_covs = np.array([[[4, 2.6], [2.6, 2.2]], [[0.3, 0], [0, 0.3]], [[1.0, -0.3], [-0.3, 0.8]]])
true_w = np.array([0.5, 0.15, 0.35])
z = rng.choice(3, 900, p=true_w)
X = np.array([rng.multivariate_normal(true_means[k], true_covs[k]) for k in z])

km = KMeans(3, n_init=10, random_state=0).fit(X)
gm = GaussianMixture(3, n_init=5, random_state=0).fit(X)
print(f"ARI so với nhóm thật: K-Means = {adjusted_rand_score(z, km.labels_):.3f} | GMM = {adjusted_rand_score(z, gm.predict(X)):.3f}")


def draw_ellipses(ax, means, covs, color="k"):
    for m, S in zip(means, covs):
        vals, vecs = np.linalg.eigh(S)
        angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        for nsig in [1, 2]:
            ax.add_patch(Ellipse(m, 2 * nsig * np.sqrt(vals[1]), 2 * nsig * np.sqrt(vals[0]), angle=angle,
                                 fill=False, color=color, lw=1.5 / nsig))


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(*X.T, c=km.labels_, cmap="tab10", s=8)
axes[0].set_title("K-Means (cụm tròn, ranh giới Voronoi)")
axes[1].scatter(*X.T, c=gm.predict(X), cmap="tab10", s=8)
draw_ellipses(axes[1], gm.means_, gm.covariances_)
axes[1].set_title("GMM (ellipse 1σ, 2σ)")
plt.show()

## 2. Mô hình sinh với biến ẩn

**Câu chuyện sinh dữ liệu**: chọn thành phần $z \sim \text{Categorical}(\pi_1, \dots, \pi_K)$, rồi sinh $\mathbf x \mid z = k \sim \mathcal N(\boldsymbol\mu_k, \Sigma_k)$. Lấy tổng theo $z$ (không quan sát được):

$$p(\mathbf x) = \sum_{k=1}^K \pi_k\,\mathcal N(\mathbf x \mid \boldsymbol\mu_k, \Sigma_k), \qquad \log p(X) = \sum_i \log\sum_k \pi_k\,\mathcal N(\mathbf x_i \mid \boldsymbol\mu_k, \Sigma_k)$$

"log của tổng" → đạo hàm theo tham số không cho nghiệm đóng. Nhưng **nếu biết $z_i$**, bài toán dễ: mỗi thành phần ước lượng MLE trên điểm của nó (như QDA, bài 01). EM khai thác điều đó.

## 3. EM from scratch

- **E-step**: với tham số hiện tại, tính **responsibility** — xác suất hậu nghiệm điểm $i$ thuộc thành phần $k$:
$$r_{ik} = \frac{\pi_k\,\mathcal N(\mathbf x_i \mid \boldsymbol\mu_k, \Sigma_k)}{\sum_j \pi_j\,\mathcal N(\mathbf x_i \mid \boldsymbol\mu_j, \Sigma_j)}$$
- **M-step**: MLE có trọng số, $N_k = \sum_i r_{ik}$:
$$\pi_k = \frac{N_k}{n}, \quad \boldsymbol\mu_k = \frac{1}{N_k}\sum_i r_{ik}\mathbf x_i, \quad \Sigma_k = \frac{1}{N_k}\sum_i r_{ik}(\mathbf x_i - \boldsymbol\mu_k)(\mathbf x_i - \boldsymbol\mu_k)^\top$$

In [ ]:
def em_gmm(X, K, n_iter=100, seed=0, reg=1e-6, tol=1e-8):
    n, d = X.shape
    r = np.random.default_rng(seed)
    means = X[r.choice(n, K, replace=False)]
    covs = np.array([np.cov(X.T) for _ in range(K)])
    weights = np.full(K, 1 / K)
    history, snapshots = [], {}
    for it in range(n_iter):
        # E-step — tính trong không gian log để tránh underflow
        log_p = np.column_stack([np.log(weights[k]) + multivariate_normal(means[k], covs[k]).logpdf(X) for k in range(K)])
        log_norm = logsumexp(log_p, axis=1)
        R = np.exp(log_p - log_norm[:, None])
        history.append(log_norm.sum())
        snapshots[it] = (means.copy(), covs.copy(), R.argmax(1))
        # M-step
        Nk = R.sum(0)
        weights = Nk / n
        means = R.T @ X / Nk[:, None]
        covs = np.array([((R[:, k, None] * (X - means[k])).T @ (X - means[k])) / Nk[k] + reg * np.eye(d) for k in range(K)])
        if it > 0 and abs(history[-1] - history[-2]) < tol * abs(history[-1]):
            break
    return {"weights": weights, "means": means, "covs": covs, "history": np.array(history), "snapshots": snapshots}


fit = em_gmm(X, 3, seed=3)
h = fit["history"]
print(f"hội tụ sau {len(h)} vòng | log-likelihood: {h[0]:.1f} → {h[-1]:.1f} | có vòng nào giảm? {(np.diff(h) < -1e-6).any()}")

In [ ]:
shots = [0, 2, 8, max(fit["snapshots"])]
fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))
for ax, it in zip(axes, shots):
    m, c, lab = fit["snapshots"][it]
    ax.scatter(*X.T, c=lab, cmap="tab10", s=5)
    draw_ellipses(ax, m, c)
    ax.set_title(f"vòng {it}: log-lik = {h[it]:.0f}")
axes[-1].plot(h, marker=".")
axes[-1].set_xlabel("vòng EM")
axes[-1].set_title("log-likelihood tăng đơn điệu")
plt.show()

In [ ]:
sk = GaussianMixture(3, n_init=5, random_state=0).fit(X)
mine = GaussianMixtureScratch(3, seed=3).fit(X)
print(f"log-lik trung bình/điểm: em_gmm = {h[-1] / len(X):.4f} | GaussianMixtureScratch = {mine.score_samples(X).mean():.4f} | sklearn = {sk.score(X):.4f}")
print("trọng số học được:", np.sort(fit["weights"]).round(3), "| thật:", np.sort(true_w))

### Vì sao EM luôn làm log-likelihood tăng? (phác thảo)

Với bất kỳ phân phối $q(z)$ nào, bất đẳng thức Jensen cho **cận dưới** (ELBO):
$$\log p(\mathbf x) = \log\sum_z q(z)\frac{p(\mathbf x, z)}{q(z)} \ge \sum_z q(z)\log\frac{p(\mathbf x, z)}{q(z)}$$
- **E-step** chọn $q(z) = p(z \mid \mathbf x)$ (responsibility) → cận dưới **chạm** log-likelihood.
- **M-step** tối đa cận dưới theo tham số → cận dưới tăng → log-likelihood (≥ cận dưới) tăng.

Nên EM tăng đơn điệu và hội tụ — nhưng tới **cực đại địa phương**. Đây là khung tổng quát cho mọi mô hình biến ẩn: HMM (bài 05), topic model, VAE (chương 34 — ELBO với $q$ là mạng nơ-ron).

## 4. Khởi tạo và cực trị địa phương

In [ ]:
finals = [em_gmm(X, 3, seed=s)["history"][-1] for s in range(12)]
print("log-likelihood cuối với 12 khởi tạo ngẫu nhiên:", np.round(finals, 1))
print(f"chênh lệch tốt nhất − tệ nhất = {max(finals) - min(finals):.1f}")

Kết quả phụ thuộc khởi tạo → chạy nhiều lần (`n_init`) và giữ log-likelihood lớn nhất; sklearn mặc định khởi tạo bằng K-Means (`init_params="kmeans"`) cho điểm xuất phát tốt.

## 5. Chọn số thành phần: BIC

Log-likelihood **luôn tăng** khi thêm thành phần → không dùng để chọn $K$. **BIC** $= -2\log\hat L + p\log n$ phạt số tham số $p$; **AIC** $= -2\log\hat L + 2p$ phạt nhẹ hơn. Chọn $K$ có BIC nhỏ nhất.

In [ ]:
Ks = range(1, 9)
bic, aic = [], []
for k in Ks:
    m = GaussianMixture(k, n_init=3, random_state=0).fit(X)
    bic.append(m.bic(X))
    aic.append(m.aic(X))
plt.plot(Ks, bic, marker="o", label="BIC")
plt.plot(Ks, aic, marker="s", label="AIC")
plt.axvline(Ks[int(np.argmin(bic))], color="r", ls="--")
plt.xlabel("số thành phần K")
plt.legend()
plt.title(f"BIC nhỏ nhất tại K = {Ks[int(np.argmin(bic))]} | AIC nhỏ nhất tại K = {Ks[int(np.argmin(aic))]}")
print(f"BIC nhỏ nhất tại K = {Ks[int(np.argmin(bic))]} | AIC nhỏ nhất tại K = {Ks[int(np.argmin(aic))]}")
plt.show()

## 6. Kiểu hiệp phương sai

| `covariance_type` | Mỗi thành phần | Tham số hiệp phương sai | Tương tự |
|---|---|---|---|
| `full` | ellipse tuỳ ý | $K\,d(d+1)/2$ | QDA |
| `tied` | cùng một ellipse | $d(d+1)/2$ | LDA |
| `diag` | ellipse song song trục | $Kd$ | Gaussian NB |
| `spherical` | hình tròn, bán kính riêng | $K$ | gần K-Means |

In [ ]:
for ct in ["full", "tied", "diag", "spherical"]:
    m = GaussianMixture(3, covariance_type=ct, n_init=3, random_state=0).fit(X)
    print(f"{ct:9s}: BIC = {m.bic(X):8.1f} | ARI = {adjusted_rand_score(z, m.predict(X)):.3f}")

Với $d$ lớn, `full` có quá nhiều tham số → `diag` hoặc `tied` (hoặc giảm chiều trước) thường an toàn hơn — chọn bằng BIC.

## 7. Failure case: suy biến (singularity)

Nếu một thành phần "co" về **một điểm** dữ liệu ($\boldsymbol\mu_k = \mathbf x_i$, $\Sigma_k = \sigma^2 I$, $\sigma \to 0$), mật độ tại điểm đó → ∞ → log-likelihood → ∞. Cực đại "tốt nhất" của GMM là vô nghĩa!

In [ ]:
x0 = X[0]
for s in [1, 0.1, 1e-2, 1e-4, 1e-8]:
    print(f"σ = {s:<6}: log N(x₀ | μ = x₀, σ²I) = {multivariate_normal(x0, s**2 * np.eye(2)).logpdf(x0):8.2f}")

Thực tế: EM thường không rơi vào đó, nhưng hay gặp khi có **điểm trùng lặp** hoặc feature hằng. Cách chữa: cộng `reg_covar` vào đường chéo (sklearn mặc định 1e-6), prior trên $\Sigma$ (MAP / `BayesianGaussianMixture`), hoặc khởi động lại thành phần suy biến.

## 8. GMM vs K-Means, và GMM như mô hình mật độ

K-Means là **trường hợp giới hạn** của EM cho GMM với $\Sigma_k = \sigma^2 I$ chung, $\sigma \to 0$: responsibility thành 0/1 (gán cứng) và M-step thành trung bình cụm.

Vì GMM là mô hình **mật độ** đầy đủ, ta có thể:

In [ ]:
gm3 = GaussianMixture(3, n_init=5, random_state=0).fit(X)
X_new, _ = gm3.sample(600)                                            # sinh dữ liệu mới
log_dens = gm3.score_samples(X)                                        # mật độ → phát hiện bất thường (chương 22)
proba = gm3.predict_proba(X)                                           # gán mềm
uncertain = (proba.max(1) < 0.8).mean()

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].scatter(*X_new.T, s=6, color="g")
axes[0].set_title("600 điểm sinh từ GMM")
axes[1].scatter(*X.T, c=log_dens, cmap="viridis", s=8)
axes[1].scatter(*X[np.argsort(log_dens)[:15]].T, s=80, facecolors="none", edgecolors="r", label="15 điểm mật độ thấp nhất")
axes[1].legend()
axes[1].set_title("log p(x) — điểm đỏ: ứng viên bất thường")
axes[2].scatter(*X.T, c=proba.max(1), cmap="magma", s=8)
axes[2].set_title(f"độ chắc chắn khi gán cụm: {uncertain:.0%} điểm có max r < 0.8")
plt.show()

## 9. Template model: Gaussian Mixture Model

| Mục | Nội dung |
|---|---|
| Problem | phân cụm mềm, ước lượng mật độ, mô hình sinh, phát hiện bất thường |
| Intuition | dữ liệu = trộn của K đám mây Gaussian |
| Math | $p(\mathbf x) = \sum_k\pi_k\mathcal N(\mathbf x \mid \boldsymbol\mu_k, \Sigma_k)$; EM tối đa log-likelihood |
| Algorithm | E-step: responsibility; M-step: MLE có trọng số; lặp tới hội tụ |
| Complexity | $O(nKd^2)$ mỗi vòng (+ $O(Kd^3)$ nghịch đảo) |
| Hyperparameters | K (BIC), `covariance_type`, `n_init`, `reg_covar`, khởi tạo |
| Assumptions | mỗi cụm xấp xỉ Gaussian (ellipse) |
| Strengths | cụm ellipse, kích thước khác nhau; xác suất gán; mật độ & sinh dữ liệu |
| Weaknesses | cực trị địa phương; suy biến; nhiều tham số khi $d$ lớn; cụm không lồi (moons) cần nhiều thành phần |
| Failure cases | singularity, $K$ sai, dữ liệu nhiều chiều với `full` |
| Production | chuẩn hoá feature, chọn K bằng BIC, nhiều khởi tạo, theo dõi log-likelihood |
| Alternatives | K-Means, DBSCAN/HDBSCAN, `BayesianGaussianMixture` (tự tắt thành phần thừa), KDE |

## 🧠 Kiểm tra nhanh

1. Viết câu chuyện sinh dữ liệu của GMM. Biến ẩn là gì?
2. Vì sao không có nghiệm đóng cho MLE của GMM?
3. E-step và M-step làm gì? Vì sao log-likelihood không giảm?
4. Vì sao không dùng log-likelihood để chọn K?
5. Singularity là gì, xử lý thế nào?
6. K-Means là trường hợp đặc biệt của GMM thế nào?

## 📌 Tóm tắt

- GMM: trộn Gaussian, biến ẩn $z$; EM: E-step responsibility, M-step MLE có trọng số; tăng đơn điệu tới cực trị địa phương.
- Nhiều khởi tạo, chọn K bằng BIC, chọn kiểu hiệp phương sai; `reg_covar` chống suy biến.
- GMM là mô hình mật độ: gán mềm, sinh dữ liệu, phát hiện bất thường.

## 📚 Tài liệu tham khảo

- Bishop (2006) — *Pattern Recognition and Machine Learning*, ch. 9
- Dempster, Laird & Rubin (1977) — *Maximum Likelihood from Incomplete Data via the EM Algorithm*
- scikit-learn User Guide — Gaussian mixture models